In [2]:
import pandas as pd
import os

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor


# ============================================================
# TRUSTSYN META-MODEL (STACKING)
# Uses existing CatBoost + D-MPNN predictions
# ============================================================


BASE = "/Users/konuri/stacking/STACKING_TABLES"


files = {
    "RANDOM": "RANDOM_STACKING_TABLE.csv",
    "COLD_COMBINATION": "COLD_COMBINATION_STACKING_TABLE.csv",
    "COLD_CELL": "COLD_CELL_STACKING_TABLE.csv"
}


results = []


for split, filename in files.items():

    print("\n====================")
    print(split)

    path = os.path.join(BASE, filename)

    df = pd.read_csv(path)

    print("Input:", df.shape)


    # -----------------------------
    # Meta features
    # -----------------------------
    X = df[
        [
            "catboost_prediction",
            "dmpnn_prediction"
        ]
    ]

    y = df["y_true"]


    models = {

        "Ridge": Ridge(
            alpha=1.0
        ),

        "XGBoost": XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            random_state=42
        )
    }


    for model_name, model in models.items():

        model.fit(
            X,
            y
        )

        pred = model.predict(X)


        rmse = mean_squared_error(
            y,
            pred
        ) ** 0.5


        r2 = r2_score(
            y,
            pred
        )


        print(
            model_name,
            "| RMSE:",
            round(rmse,4),
            "| R2:",
            round(r2,4)
        )


        results.append(
            {
                "split": split,
                "model": model_name,
                "RMSE": rmse,
                "R2": r2
            }
        )


    # Save meta input table
    df[
        [
            "drug_A",
            "drug_B",
            "CELLNAME",
            "y_true",
            "catboost_prediction",
            "dmpnn_prediction"
        ]
    ].to_csv(
        f"{BASE}/{split}_META_INPUT.csv",
        index=False
    )


# ============================================================
# SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

print("\n====================")
print(results_df)


results_df.to_csv(
    f"{BASE}/META_MODEL_RESULTS.csv",
    index=False
)


print("\nDONE")


RANDOM
Input: (29408, 74)
Ridge | RMSE: 5.888 | R2: 0.2944
XGBoost | RMSE: 5.9109 | R2: 0.2889

COLD_COMBINATION
Input: (29558, 74)
Ridge | RMSE: 6.5782 | R2: 0.1266
XGBoost | RMSE: 6.5682 | R2: 0.1293

COLD_CELL
Input: (29787, 74)
Ridge | RMSE: 7.0544 | R2: 0.2689
XGBoost | RMSE: 6.789 | R2: 0.3229

              split    model      RMSE        R2
0            RANDOM    Ridge  5.888001  0.294404
1            RANDOM  XGBoost  5.910889  0.288907
2  COLD_COMBINATION    Ridge  6.578213  0.126644
3  COLD_COMBINATION  XGBoost  6.568226  0.129294
4         COLD_CELL    Ridge  7.054416  0.268890
5         COLD_CELL  XGBoost  6.788992  0.322872

DONE


In [3]:
import os
from pathlib import Path

for folder in [
    "/Users/konuri/stacking/CATBOOST_TEAM_PACKAGE",
    "/Users/konuri/stacking/TrustSyn_DMPNN"
]:

    print("\nSEARCHING:", folder)

    for f in Path(folder).rglob("*prediction*.csv"):
        print(f)


SEARCHING: /Users/konuri/stacking/CATBOOST_TEAM_PACKAGE
/Users/konuri/stacking/CATBOOST_TEAM_PACKAGE/COLD_DRUG_CatBoost_predictions.csv

SEARCHING: /Users/konuri/stacking/TrustSyn_DMPNN
/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_CELL_LINE/final_v2_c_test_predictions.csv
/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_COMBINATION/final_v2_c_test_predictions.csv
/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/RANDOM/final_v2_c_test_predictions.csv
/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_DRUG/final_v2_c_test_predictions.csv
